In [1]:
"""
台北市門牌資料爬蟲 — 完整流程
直接在 Jupyter 逐 cell 執行，或整個跑完
"""

# ── Cell 1：初始化 session ──────────────────────────────
from __future__ import annotations
import requests
import pandas as pd
import re
import json
import time
import sqlite3
import os
from bs4 import BeautifulSoup
from PIL import Image as PILImage
from IPython.display import display, Image

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-TW,zh;q=0.9,en;q=0.8",
    "Referer": "https://www.ris.gov.tw/info-doorplate/app/doorplate/main"
})

CITY_CODE = "63000000"   # 台北市
DB_PATH   = "taiwan_address.db"
CSV_DIR   = "csv_backup"
os.makedirs(CSV_DIR, exist_ok=True)

print("✅ Session 初始化完成")



✅ Session 初始化完成


In [33]:
# ── Cell 2：進主頁 + 取得行政區列表 ───────────────────────
def setup_and_get_districts():
    """
    進主頁 → 進 map → 進 query 頁
    同時取得：current_csrf、captchaKey、df_districts
    回傳 (csrf, captcha_key, df_districts, response_query)
    """
    # Step 1: 進主頁（讓 session 拿到 cookie）
    session.get("https://www.ris.gov.tw/info-doorplate/app/doorplate/main")

    # Step 2: POST map 頁（searchType=date）
    response_map = session.post(
        "https://www.ris.gov.tw/info-doorplate/app/doorplate/map",
        data={"searchType": "date"}
    )

    # Step 3: 抓 CSRF（先找 input，找不到找 meta，都沒有就用空字串）
    soup_map = BeautifulSoup(response_map.text, "html.parser")
    csrf_input = soup_map.find("input", {"name": "_csrf"})
    csrf_meta  = soup_map.find("meta",  {"name": "_csrf"})

    if csrf_input:
        csrf = csrf_input["value"]
    elif csrf_meta:
        csrf = csrf_meta["content"]
    else:
        csrf = ""   # 有些 session 不需要 csrf 也能過
        print("⚠️  找不到 CSRF，用空字串繼續（通常不影響）")

#     print(f"csrf = {csrf!r}")

    # Step 4: POST query 頁（選台北市）
    response_query = session.post(
        "https://www.ris.gov.tw/info-doorplate/app/doorplate/query",
        data={
            "searchType": "date",
            "cityCode":   CITY_CODE,
            "_csrf":      csrf
        }
    )
#     print(response_query.url)

    soup_query = BeautifulSoup(response_query.text, "html.parser")

    csrf_input = soup_query.find("input", {"name": "_csrf"})
    csrf_meta  = soup_query.find("meta", {"name": "_csrf"})

#     soup_query = BeautifulSoup(response_query.text, "html.parser")

    # Step 5: 抓行政區列表
    area_select = soup_query.find("select", {"name": "areaCode"})
    districts   = []
    for opt in area_select.find_all("option"):
        code = opt.get("value", "").strip()
        name = opt.get_text(strip=True)
        if code and "選擇" not in name:
            districts.append({"區名": name, "區代碼": code})
    df_districts = pd.DataFrame(districts)

    # Step 6: 抓 captchaKey
    captcha_hidden = soup_query.find("input", {"name": "captchaKey"})
    captcha_key    = captcha_hidden["value"] if captcha_hidden else ""

    print(f"✅ 行政區共 {len(df_districts)} 個，captchaKey={captcha_key!r}")
    return csrf, captcha_key, df_districts



csrf, captcha_key, df_districts = setup_and_get_districts()
print(df_districts)


✅ 行政區共 12 個，captchaKey='f8f5c6ecc4ed4bc99e46c25f172ab2c2'
     區名       區代碼
0   松山區  63000010
1   信義區  63000020
2   大安區  63000030
3   中山區  63000040
4   中正區  63000050
5   大同區  63000060
6   萬華區  63000070
7   文山區  63000080
8   南港區  63000090
9   內湖區  63000100
10  士林區  63000110
11  北投區  63000120


In [3]:
# ──顯示驗證碼 ────────────────────────────────────
def show_captcha(captcha_key: str) -> bytes:
    """下載驗證碼圖片並在 Jupyter 顯示，回傳圖片 bytes"""
    url = (
        "https://www.ris.gov.tw/info-doorplate/captcha/image"
        f"?CAPTCHA_KEY={captcha_key}&time={int(time.time()*1000)}"
    )
    img_bytes = session.get(url).content
    with open("captcha.jpg", "wb") as f:
        f.write(img_bytes)
    display(Image(filename="captcha.jpg"))
    return img_bytes

show_captcha(captcha_key)

b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t\x0c\x0b\x0c\x18\r\r\x182!\x1c!22222222222222222222222222222222222222222222222222\xff\xc0\x00\x11\x08\x00\x1c\x00P\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xc2\xc3\xc4\xc5\xc6\xc7\xc8\xc9\xca\xd2\xd3\xd4

In [41]:
# ── Cell 4：查詢單一行政區 ───────────────────────────────
def query_area(
    area_code:     str,
    captcha_input: str,
    captcha_key:   str,
    csrf:          str,
    city_code:     str = CITY_CODE,
    date_from:     str = "114-09-01",
    date_to:       str = "114-11-30",
) -> dict:
    """打 inquiry API，回傳 JSON dict"""
    payload = {
        "searchType":    "date",
        "cityCode":      city_code,
        "tkt":           "-1",
        "areaCode":      area_code,
        "village":       "",
        "neighbor":      "",
        "sDate":         date_from,
        "eDate":         date_to,
        "_includeNoDate": "on",
        "registerKind":  "1",
        "captchaInput":  captcha_input,
        "captchaKey":    captcha_key,
        "_csrf":         csrf,
        "_search":       "false",
        "draw": 1,
        "start": 0,
        "length": 999,
#         "rows":          "9999",
#         "page":          "1",
        "sord":          "asc"
    }
    r = session.post(
        "https://www.ris.gov.tw/info-doorplate/app/doorplate/inquiry/date",
        data=payload
    )


    return r.json()


In [5]:
# ── 資料清理 ──────────────────────────────────────
def clean_date(roc_date_str: str) -> str:
    """114/09/01 → 2025-09-01"""
    nums = re.findall(r'\d+', roc_date_str)
    if len(nums) == 3:
        return f"{int(nums[0])+1911}-{int(nums[1]):02d}-{int(nums[2]):02d}"
    return roc_date_str


def clean_address(full_address: str, city: str = "臺北市") -> dict:
    """解析完整門牌地址成結構化欄位"""
    # 動態城市名（相容台/臺）
    city_pattern = city.replace("台", "[台臺]").replace("臺", "[台臺]")
    pattern = re.compile(
        rf'(?P<city>{city_pattern})'
        r'(?P<district>[^\u5340]+\u5340)'   # XXX區
        r'(?P<village>[^\u91cc]+\u91cc)?'   # XXX里（optional）
        r'(?P<neighborhood>\d+\u9130)?'     # N鄰（optional）
        r'(?P<detail>.*)'
    )
    match = pattern.match(full_address)
    if match:
        d = match.groupdict()
        d["village"]      = d["village"]      or "未分類"
        d["neighborhood"] = d["neighborhood"] or "未分類"
        return d
    return {
        "city": city, "district": "未知區",
        "village": "未分類", "neighborhood": "未分類",
        "detail": full_address
    }


In [38]:
def get_next_captcha(data: dict):
    """從回應取得下一個 captchaKey 和 csrf token"""
    try:
        msg = json.loads(data.get("message", "{}"))
        return msg.get("captcha"), msg.get("token")
    except Exception:
        return None, None


def save_area(area_name: str, data: dict, city: str = "臺北市"):
    rows = data.get("data", [])  # ← 改這裡，原本是 "rows"
    if not rows:
        print(f"❌ {area_name} 無資料")
        return 0

    records = []
    for row in rows:
        addr = clean_address(row["v1"], city=city)
        records.append({
            "city":         addr.get("city", city),
            "district":     addr.get("district", area_name),
            "village":      addr.get("village", "未分類"),
            "neighborhood": addr.get("neighborhood", "未分類"),
            "detail":       addr.get("detail", ""),
            "編釘日期":     clean_date(row["v2"]),
            "編釘類別":     row["v3"],
            "原始門牌地址": row["v1"],
        })

    df = pd.DataFrame(records)
    csv_path = os.path.join(CSV_DIR, f"{area_name}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
#     print(f"  💾 CSV：{csv_path}")

    conn = sqlite3.connect(DB_PATH, timeout=30.0)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS taiwan_handicraft (
            city TEXT, district TEXT, village TEXT,
            neighborhood TEXT, detail TEXT,
            編釘日期 TEXT, 編釘類別 TEXT, 原始門牌地址 TEXT,
            PRIMARY KEY (city, district, detail, 編釘日期)
        )
    """)
    for r in records:
        conn.execute("""
            INSERT OR IGNORE INTO taiwan_handicraft
            VALUES (:city,:district,:village,:neighborhood,
                    :detail,:編釘日期,:編釘類別,:原始門牌地址)
        """, r)
    conn.commit()
    conn.close()
    print(f"  🗄️  SQLite 寫入：{len(records)} 筆")
    return len(records)


def run_all_districts(df_districts, csrf, captcha_key):
#     current_key  = captcha_key
#     current_csrf = csrf
    results = []

    for i, row in df_districts.iterrows():

        # 每區重新取得
        current_csrf, current_key, _ = setup_and_get_districts()

        area_name = row["區名"]
        area_code = row["區代碼"]

        print(f"\n{'='*50}")
        print(f"[{i+1}/{len(df_districts)}] 查詢：{area_name}")

        while True:
#             show_captcha(current_key)
#             captcha_input = input(f"請輸入「{area_name}」驗證碼：").strip()
            print(f"\n=== {area_name} ===")
#             print("cookies =", session.cookies.get_dict())
            print("captcha_key =", current_key)
            print("csrf =", current_csrf)

            show_captcha(current_key)

            captcha_input = input(
                f"請輸入「{area_name}」驗證碼："
            ).strip()

     
            data = query_area(
                area_code     = area_code,
                captcha_input = captcha_input,
                captcha_key   = current_key,
                csrf          = current_csrf,
            )


            # 正確判斷成功：用 recordsTotal 和 success
            success      = data.get("success", False)
            record_count = data.get("recordsTotal", 0)

            if not success or record_count == 0:
                print(f"❌ 失敗（success={success}, records={record_count}），重新 setup")
                current_csrf, current_key, _ = setup_and_get_districts()
                retry = input("重試？(y/n)：").strip().lower()
                if retry != "y":
                    results.append({"區名": area_name, "狀態": "跳過", "筆數": 0})
                    break
                continue

            count = save_area(area_name, data)
            results.append({"區名": area_name, "狀態": "成功", "筆數": count})
            print(f"✅ {area_name} 完成，共 {count} 筆")
            break

        time.sleep(1)

    print(f"\n{'='*50}")
    print(pd.DataFrame(results).to_string(index=False))
    return results

In [44]:
pythoncsrf, captcha_key, df_districts = setup_and_get_districts()
results = run_all_districts(df_districts, csrf, captcha_key)

✅ 行政區共 12 個，captchaKey='b89bd6c7ef2b41deb559b6eed9d2c7cf'
✅ 行政區共 12 個，captchaKey='022f8ca606c246babdaed6618b194624'

[1/12] 查詢：松山區

=== 松山區 ===
captcha_key = 022f8ca606c246babdaed6618b194624
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「松山區」驗證碼：EZJUI

=== BEFORE UPDATE ===
current_key = 022f8ca606c246babdaed6618b194624
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = 022f8ca606c246babdaed6618b194624
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

❌ 失敗（success=False, records=0），重新 setup
✅ 行政區共 12 個，captchaKey='65dbd28cf984417faeb5413288f39c72'
重試？(y/n)：N
✅ 行政區共 12 個，captchaKey='5ddc793b52ce4a288797faeab9ee12a8'

[2/12] 查詢：信義區

=== 信義區 ===
captcha_key = 5ddc793b52ce4a288797faeab9ee12a8
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「信義區」驗證碼：YJTP6

=== BEFORE UPDATE ===
current_key = 5ddc793b52ce4a288797faeab9ee12a8
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = 5ddc793b52ce4a288797faeab9ee12a8
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

recordsTotal = 124
len(data) = 124
  💾 CSV：csv_backup\信義區.csv
  🗄️  SQLite 寫入：124 筆
✅ 信義區 完成，共 124 筆
✅ 行政區共 12 個，captchaKey='acc4c9cc3d114f53bc411f44b8be5dea'

[3/12] 查詢：大安區

=== 大安區 ===
captcha_key = acc4c9cc3d114f53bc411f44b8be5dea
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「大安區」驗證碼：H74HZ

=== BEFORE UPDATE ===
current_key = acc4c9cc3d114f53bc411f44b8be5dea
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = acc4c9cc3d114f53bc411f44b8be5dea
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

recordsTotal = 130
len(data) = 130
  💾 CSV：csv_backup\大安區.csv
  🗄️  SQLite 寫入：130 筆
✅ 大安區 完成，共 130 筆
✅ 行政區共 12 個，captchaKey='92405e8add314264829df45a0f84ba4a'

[4/12] 查詢：中山區

=== 中山區 ===
captcha_key = 92405e8add314264829df45a0f84ba4a
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「中山區」驗證碼：DUUGR

=== BEFORE UPDATE ===
current_key = 92405e8add314264829df45a0f84ba4a
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = 92405e8add314264829df45a0f84ba4a
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

recordsTotal = 904
len(data) = 904
  💾 CSV：csv_backup\中山區.csv
  🗄️  SQLite 寫入：904 筆
✅ 中山區 完成，共 904 筆
✅ 行政區共 12 個，captchaKey='b84d725c4ce3454f87169ec05bd96457'

[5/12] 查詢：中正區

=== 中正區 ===
captcha_key = b84d725c4ce3454f87169ec05bd96457
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


...

請輸入「文山區」驗證碼：H79YP

=== BEFORE UPDATE ===
current_key = 24bae98a482f483e8159de92a333f5fc
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = 24bae98a482f483e8159de92a333f5fc
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

recordsTotal = 170
len(data) = 170
  💾 CSV：csv_backup\文山區.csv
  🗄️  SQLite 寫入：170 筆
✅ 文山區 完成，共 170 筆
✅ 行政區共 12 個，captchaKey='bd3d388b796d491a944dd41d2906da11'

[9/12] 查詢：南港區

=== 南港區 ===
captcha_key = bd3d388b796d491a944dd41d2906da11
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「南港區」驗證碼：6HVUQ

=== BEFORE UPDATE ===
current_key = bd3d388b796d491a944dd41d2906da11
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = bd3d388b796d491a944dd41d2906da11
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

❌ 失敗（success=False, records=0），重新 setup
✅ 行政區共 12 個，captchaKey='a1a4ef2a37994928a5da18e16eea843a'
重試？(y/n)：N
✅ 行政區共 12 個，captchaKey='dc7d295f1ba74c3cb7f1148ab162e3dc'

[10/12] 查詢：內湖區

=== 內湖區 ===
captcha_key = dc7d295f1ba74c3cb7f1148ab162e3dc
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「內湖區」驗證碼：BKAEU

=== BEFORE UPDATE ===
current_key = dc7d295f1ba74c3cb7f1148ab162e3dc
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = dc7d295f1ba74c3cb7f1148ab162e3dc
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

recordsTotal = 106
len(data) = 106
  💾 CSV：csv_backup\內湖區.csv
  🗄️  SQLite 寫入：106 筆
✅ 內湖區 完成，共 106 筆
✅ 行政區共 12 個，captchaKey='6239f607f2ce4881851ef0f9ad6859f8'

[11/12] 查詢：士林區

=== 士林區 ===
captcha_key = 6239f607f2ce4881851ef0f9ad6859f8
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「士林區」驗證碼：4D5PN

=== BEFORE UPDATE ===
current_key = 6239f607f2ce4881851ef0f9ad6859f8
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = 6239f607f2ce4881851ef0f9ad6859f8
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

recordsTotal = 319
len(data) = 319
  💾 CSV：csv_backup\士林區.csv
  🗄️  SQLite 寫入：319 筆
✅ 士林區 完成，共 319 筆
✅ 行政區共 12 個，captchaKey='c423da7b49d741ec96f670b19cc65b5c'

[12/12] 查詢：北投區

=== 北投區 ===
captcha_key = c423da7b49d741ec96f670b19cc65b5c
csrf = d82fd098-73a6-4aee-bcef-1520176d0fda


請輸入「北投區」驗證碼：ZG9JX

=== BEFORE UPDATE ===
current_key = c423da7b49d741ec96f670b19cc65b5c
current_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda
updated_key = c423da7b49d741ec96f670b19cc65b5c
updated_csrf = d82fd098-73a6-4aee-bcef-1520176d0fda

recordsTotal = 405
len(data) = 405
  💾 CSV：csv_backup\北投區.csv
  🗄️  SQLite 寫入：405 筆
✅ 北投區 完成，共 405 筆

 區名 狀態  筆數
松山區 跳過   0
信義區 成功 124
大安區 成功 130
中山區 成功 904
中正區 成功 213
大同區 成功 459
萬華區 成功 111
文山區 成功 170
南港區 跳過   0
內湖區 成功 106
士林區 成功 319
北投區 成功 405


In [30]:
# csrf, captcha_key, df_districts = setup_and_get_districts()
# show_captcha(captcha_key)

[('中山區', 904), ('中正區', 213), ('信義區', 124), ('內湖區', 106), ('北投區', 405), ('士林區', 319), ('大同區', 459), ('大安區', 130), ('文山區', 170), ('松山區', 246), ('萬華區', 111)]
